# BistBull Phase 4.7+ — Production Smoke Test

Tek hücrede production deploy doğrulaması. Push sonrası Railway redeploy bittikten sonra (~2-3 dk) bunu çalıştır:

1. Calibrated path aktif mi? (`scoring_version_effective='calibrated_2026Q1'`)
2. K3 (Türkiye gerçekleri) çalışıyor mu? (`composite_multiplier`)
3. K4 (academic) çalışıyor mu? (`academic_penalty`)

3/3 geçer → calibrated_2026Q1 production'da canlı.

Test sembolü değiştirebilirsin: THYAO yerine başka bir liquid BIST sembolü (AKBNK, ASELS, EREGL, vs.).

In [ ]:
# ============================================================
# Production smoke test — pure stdlib, no zip needed
# ============================================================
import urllib.request, urllib.error, json, sys

URL = 'https://bistbull.ai'  # production URL — değiştirme
SYMBOL = 'THYAO'             # değiştirebilirsin
TIMEOUT = 30                  # saniye

endpoint = f'{URL}/api/analyze/{SYMBOL}?scoring_version=calibrated_2026Q1'
print(f'GET {endpoint}')
print(f'(timeout: {TIMEOUT}s)')
print()

try:
    req = urllib.request.Request(endpoint, headers={'User-Agent': 'bistbull-smoke/1.0'})
    with urllib.request.urlopen(req, timeout=TIMEOUT) as resp:
        if resp.status != 200:
            print(f'❌ HTTP {resp.status}')
            sys.exit(1)
        body = json.loads(resp.read().decode('utf-8'))
except urllib.error.HTTPError as e:
    print(f'❌ HTTP error: {e.code} {e.reason}')
    print('   Railway deploy henüz bitmemiş olabilir. 1-2 dk bekle, tekrar dene.')
    sys.exit(1)
except urllib.error.URLError as e:
    print(f'❌ Connection error: {e.reason}')
    sys.exit(1)
except Exception as e:
    print(f'❌ Unexpected error: {type(e).__name__}: {e}')
    sys.exit(1)

# ============================================================
# Üç kontrol
# ============================================================
checks = []
data = body.get('data', {}) if isinstance(body, dict) else {}
meta = data.get('_meta', {})
scores = data.get('scores', data) if isinstance(data, dict) else {}

# 1. Calibrated path active?
ver = meta.get('scoring_version_effective')
if ver == 'calibrated_2026Q1':
    checks.append(('1. calibrated_2026Q1 active', True, ver))
else:
    checks.append(('1. calibrated_2026Q1 active', False,
                   f'got {ver!r} (expected calibrated_2026Q1)'))

# 2. K3 turkey realities composite multiplier present?
tr = (data.get('turkey_realities')
      or data.get('layers', {}).get('turkey_realities')
      or {})
cm = tr.get('composite_multiplier') if isinstance(tr, dict) else None
if cm is not None:
    checks.append(('2. K3 composite_multiplier', True, f'{cm:.4f}'))
else:
    checks.append(('2. K3 composite_multiplier', False, 'missing'))

# 3. K4 academic penalty present?
ac = (data.get('academic')
      or data.get('layers', {}).get('academic')
      or {})
ap = ac.get('academic_penalty') if isinstance(ac, dict) else None
if ap is not None:
    checks.append(('3. K4 academic_penalty', True, f'{ap:.4f}'))
else:
    checks.append(('3. K4 academic_penalty', False, 'missing'))

# ============================================================
# Sonuç
# ============================================================
print('=' * 60)
for name, ok, val in checks:
    icon = '✅' if ok else '❌'
    print(f'{icon} {name}: {val}')
print('=' * 60)

n_ok = sum(1 for _, ok, _ in checks if ok)
n_total = len(checks)
if n_ok == n_total:
    print(f'\n✅ {n_ok}/{n_total} kontrol başarılı — calibrated_2026Q1 production\'da canlı.')
else:
    print(f'\n❌ {n_ok}/{n_total} kontrol başarılı.')
    print(f'\nResponse JSON ilk 800 karakter (debug için):')
    print(json.dumps(body, indent=2)[:800])
    sys.exit(1)

## A/B telemetry quick check (opsiyonel)

Production'da paired snapshot'ların biriktiğini doğrula:

In [ ]:
import urllib.request, json
URL = 'https://bistbull.ai'
endpoint = f'{URL}/api/scoring/ab_report_breakdown?days=7'
print(f'GET {endpoint}')

try:
    with urllib.request.urlopen(endpoint, timeout=20) as r:
        d = json.loads(r.read().decode('utf-8'))
    m = d.get('_meta', {})
    print(f"\nPaired rows: {m.get('n_paired_rows')}")
    print(f"Sembol: {m.get('n_symbols')}, Sektör: {m.get('n_sectors')}")
    
    by_sym = d.get('by_symbol', [])
    if by_sym:
        print(f"\nEn divergent 5 sembol:")
        for r in by_sym[:5]:
            print(f"  {r['symbol']:8} ({r['sector']:12}) max_abs={r['max_abs_diff']:+.2f}, flips={r['decision_flips']}")
    else:
        print('\nHenüz paired snapshot birikmemiş — 1-2 gün bekle.')
except Exception as e:
    print(f'Endpoint çağrısı başarısız: {type(e).__name__}: {e}')